# StatCore k-NN: prototype compression, learnable metric and data-adaptive k

[Open in Colab](https://colab.research.google.com/github/Jabualani/ML-Reliabilty/blob/main/docs/colab/statcore-knn.ipynb)

**Paper topic:** StatCore Nearest Neighbours: Exact Baselines, Prototype Compression, Diagonal Metric Learning, and Data-Adaptive k — `docs/latex/statcore-knn.tex` ([repository](https://github.com/Jabualani/ML-Reliabilty/blob/main/docs/latex/statcore-knn.tex)).
**Experiment script reproduced here:** `docs/latex/experiments/knn_experiment.py` (reduced seed count so the notebook runs in a few minutes).

StatCore is a pure-NumPy/SciPy model family in `ml_reliability.statcore`. Every model has a `variant`
argument; each variant below is compared with the standard reference implementation.
**scikit-learn is used only as the baseline/reference** (and for loading public datasets), never inside StatCore.

## 1. Install (parameter cell)

> **Note:** StatCore has **not been pushed / published yet**. Until the repository is public, the
> `git+https` install below will fail. In that case use the **fallback cell**: zip the repository folder,
> upload the zip to Colab (left sidebar → Files, or the `files.upload()` helper) and `pip install` it.
> Run exactly one of the two cells.

In [ ]:
# @title Install StatCore (PARAMETER: change the source if needed)
# Option A - once the repository is published:
!pip install "git+https://github.com/Jabualani/ML-Reliabilty.git"

In [ ]:
# @title FALLBACK install from an uploaded zip of the repository (use if Option A fails)
# Zip the repo folder (ML-Reliabilty.zip), then uncomment and run:
# from google.colab import files
# files.upload()                       # pick ML-Reliabilty.zip
# !unzip -q -o ML-Reliabilty.zip -d repo
# !pip install ./repo/ML-Reliabilty     # adjust to the folder that contains pyproject.toml

In [ ]:
# @title Parameters and imports
SEEDS = (11, 23)   # the paper uses more seeds; fewer here to keep the run short

import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
np.set_printoptions(precision=4, suppress=True)
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30)

## 2. Method summary

All variants standardise features, $z=(x-m)/s$, and predict by a (weighted) average of the targets of the $k$ nearest references under $d^2_a(z,z')=\sum_j a_j(z_j-z'_j)^2$ (with $a=\mathbf 1$ unless the metric is learned):
$$\hat t(z)=\frac{\sum_{j\in N_k(z)} r_j\,\omega_j\,t_j}{\sum_{j\in N_k(z)} r_j\,\omega_j},\qquad \omega_j=1\ \text{(uniform)}\ \text{or}\ 1/(d_j+10^{-9})\ \text{(distance)},$$
where $t_j$ is a one-hot label (classification; $\hat t$ are the class probabilities) or the response (regression) and $r_j$ are reference weights ($r_j=1$ except for prototypes). Reference: sklearn `Pipeline(StandardScaler, KNeighbors*(n_neighbors=5))`.

**`base`** — exact Euclidean k-NN with $k=$ `n_neighbors`.

**`prototype_compressed`** — per class (classification) or over the whole set (regression, clustering in $(z, \text{standardised } y)$), k-means (`better_init`, 2 restarts, 50 iterations) with $m=\lceil\gamma\,n_c\rceil$ clusters, $\gamma=$ `compression`. Each cluster $S$ becomes a prototype at the mean of its members' $z$ with target equal to the mean of its targets and reference weight $r=|S|$. Classes with $m\ge n_c$ keep all points.

**`learnable_metric`** — diagonal metric $a=e^{\theta}$ learned by Adam on a random sub-sample of at most 400 points. With soft neighbours $p_{ij}=\mathrm{softmax}_{j\ne i}\big(-d^2_a(z_i,z_j)\big)$:
- classification maximises the leave-one-out NCA objective $\frac1m\sum_i q_i$, $q_i=\sum_{j:\,y_j=y_i}p_{ij}$;
- regression minimises $\frac1m\sum_i(\hat y_i-y_i)^2$ with $\hat y_i=\sum_j p_{ij}y_j$ (standardised $y$).

Adam ($\beta=(0.9,0.999)$, `metric_lr`, `metric_iters` steps) acts on $\theta$ clipped to $[-8,3]$; the final metric is normalised to $\mathrm{mean}(a)=1$. Irrelevant features should receive small $a_j$.

**`stat_k_selection`** — leave-one-out (self excluded) over odd $k\le30$:
$$L_k=\Big(1-\mathrm{acc}_k\Big)+10^{-3}\,\overline{-\log\max(\hat p_{y},10^{-3})}\quad(\text{classification}),\qquad L_k=\overline{(\hat y-y)^2}\quad(\text{regression}),$$
and the smallest minimiser is used (ties go to the smaller $k$).

## 3. Experiment (mirrors `knn_experiment.py` / `_experiment_utils.py`)

Classification and regression, each on a *synthetic nonlinear* set (2 relevant + 18 irrelevant features, 640 rows, 10 % label flips for classification) and a real sklearn set (breast cancer / diabetes). 75/25 split (stratified for classification); $k=5$, uniform weights, `compression=0.35`, 40 metric iterations.

In [ ]:
from sklearn.datasets import load_breast_cancer, load_diabetes
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def _synthetic(kind, seed, n=640):
    rng = np.random.default_rng(seed + (10000 if kind == "regression" else 0))
    x = rng.uniform(-2.0, 2.0, size=(n, 2)); noise = rng.normal(size=n)
    if kind == "classification":
        y = ((x[:, 0] ** 2 + 0.65 * x[:, 1] ** 2 + 0.25 * noise) > 1.65).astype(int)
        flip = rng.random(n) < 0.10; y[flip] = 1 - y[flip]
    else:
        y = np.sin(1.4 * x[:, 0]) + 0.35 * x[:, 1] ** 2 + 0.30 * noise
    return np.column_stack((x, rng.normal(size=(n, 18)))), y      # 2 relevant + 18 irrelevant features

def datasets(kind, seed):
    X, y = _synthetic(kind, seed)
    yield "synthetic_nonlinear_irrelevant", X, y
    bunch = load_breast_cancer() if kind == "classification" else load_diabetes()
    yield ("sklearn_breast_cancer" if kind == "classification" else "sklearn_diabetes"), bunch.data, bunch.target

def split_data(X, y, seed, stratify):
    rng = np.random.default_rng(seed); n_test = int(np.ceil(0.25 * len(y)))
    if not stratify:
        order = rng.permutation(len(y))
        return X[order[n_test:]], X[order[:n_test]], y[order[n_test:]], y[order[:n_test]]
    classes, inverse, counts = np.unique(y, return_inverse=True, return_counts=True)
    raw = counts * (n_test / len(y)); allocation = np.floor(raw).astype(int)
    remaining = n_test - int(allocation.sum()); tie_break = rng.random(len(counts))
    order = np.lexsort((tie_break, -(raw - allocation))); allocation[order[:remaining]] += 1
    train_ids, test_ids = [], []
    for c, count in enumerate(allocation):
        ids = rng.permutation(np.flatnonzero(inverse == c)); test_ids.extend(ids[:count]); train_ids.extend(ids[count:])
    train_ids = rng.permutation(np.asarray(train_ids, dtype=int)); test_ids = rng.permutation(np.asarray(test_ids, dtype=int))
    return X[train_ids], X[test_ids], y[train_ids], y[test_ids]

def ece_equal_width(y, prob, bins=15):
    pred = prob.argmax(axis=1); conf = prob.max(axis=1); correct = (pred == y).astype(float)
    edges = np.linspace(0.0, 1.0, bins + 1); ans = 0.0
    for j in range(bins):
        mask = (conf >= edges[j]) & ((conf < edges[j + 1]) if j < bins - 1 else (conf <= edges[j + 1]))
        if mask.any():
            ans += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return float(ans)

def classification_scores(y, prob):
    prob = np.clip(np.asarray(prob), 1e-12, 1.0); prob = prob / prob.sum(axis=1, keepdims=True)
    truth = np.searchsorted(np.unique(y), y)
    return {"accuracy": float(np.mean(truth == prob.argmax(axis=1))),
            "log_loss": float(-np.log(prob[np.arange(len(y)), truth]).mean()),
            "ece": ece_equal_width(truth, prob)}

def regression_scores(y, pred):
    residual = np.asarray(y) - np.asarray(pred); total = np.sum((y - np.mean(y)) ** 2)
    return {"mse": float(np.mean(residual ** 2)), "mae": float(np.mean(np.abs(residual))),
            "r2": float(1.0 - np.sum(residual ** 2) / total)}

def run_family(make_models, seeds):
    import time
    rows = []
    for task in ("classification", "regression"):
        for seed in seeds:
            for dataset, X, y in datasets(task, seed):
                Xtr, Xte, ytr, yte = split_data(X, y, seed=seed, stratify=(task == "classification"))
                for name, model in make_models(task, seed).items():
                    start = time.perf_counter(); model.fit(Xtr, ytr)
                    if task == "classification":
                        prob = model.predict_proba(Xte); classes = model.classes_
                        if not np.array_equal(classes, np.arange(len(classes))):
                            prob = prob[:, [int(np.flatnonzero(classes == c)[0]) for c in np.arange(len(classes))]]
                        metrics = classification_scores(yte, prob)
                    else:
                        metrics = regression_scores(yte, model.predict(Xte))
                    rows.append({"task": task, "dataset": dataset, "seed": seed, "model": name,
                                 "runtime_s": time.perf_counter() - start, **metrics})
    return pd.DataFrame(rows)

def summarise(res, order):
    for task, metrics in (("classification", ["accuracy", "log_loss", "ece", "runtime_s"]),
                          ("regression", ["mse", "mae", "r2", "runtime_s"])):
        sub = res[res.task == task]
        g = sub.groupby(["dataset", "model"], sort=False)[metrics]
        m, s = g.mean(), g.std()
        out = m.round(4).astype(str) + " +/- " + s.round(4).astype(str)
        print(f"\n[{task}] mean +/- sample SD over seeds"); print(out.loc[pd.MultiIndex.from_product([sub.dataset.unique(), order])].to_string())
    return res

from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor      # baseline only
from ml_reliability.statcore import KNNClassifier, KNNRegressor

VARIANT_NAMES = ("base", "prototype_compressed", "learnable_metric", "stat_k_selection")

def make_models(task, seed):
    is_class = task == "classification"
    baseline = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5) if is_class else KNeighborsRegressor(n_neighbors=5))
    cls = KNNClassifier if is_class else KNNRegressor
    models = {name: cls(variant=name, n_neighbors=5, weights="uniform", compression=0.35,
                        metric_iters=40, metric_lr=0.1, random_state=seed) for name in VARIANT_NAMES}
    return {"sklearn_standard": baseline, **models}

ORDER = ["sklearn_standard", *VARIANT_NAMES]
res = run_family(make_models, SEEDS)
print(f"k-NN experiment over seeds {SEEDS} (runtime = single fit + predict, no warm-up)")
res = summarise(res, ORDER)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
for ax, (task, metric) in zip(axes, (("classification", "accuracy"), ("regression", "mse"))):
    t = res[res.task == task].groupby(["dataset", "model"])[metric].mean().unstack("model")[ORDER]
    t.plot.bar(ax=ax, rot=15, legend=(task == "regression")); ax.set_title(f"{task}: {metric}")
    if metric == "mse": ax.set_yscale("log")
plt.tight_layout(); plt.show()

## 4. Look inside each variant

In [ ]:
seed = SEEDS[0]
Xtr, Xte, ytr, yte = split_data(*_synthetic("classification", seed), seed=seed, stratify=True)
lm = KNNClassifier(variant="learnable_metric", metric_iters=40, random_state=seed).fit(Xtr, ytr)
sk = KNNClassifier(variant="stat_k_selection", random_state=seed).fit(Xtr, ytr)
pc = KNNClassifier(variant="prototype_compressed", compression=0.35, random_state=seed).fit(Xtr, ytr)

fig, axes = plt.subplots(1, 3, figsize=(15, 3.4))
axes[0].bar(range(20), lm.metric_diag_, color=["C3"] * 2 + ["C0"] * 18)
axes[0].set_title("classification: learned a_j (red = 2 relevant)"); axes[0].set_xlabel("feature")
Rtr, Rte, rytr, ryte = split_data(*_synthetic("regression", seed), seed=seed, stratify=False)
lmr = KNNRegressor(variant="learnable_metric", metric_iters=40, random_state=seed).fit(Rtr, rytr)
axes[1].bar(range(20), lmr.metric_diag_, color=["C3"] * 2 + ["C0"] * 18)
axes[1].set_title("regression: learned a_j (red = 2 relevant)"); axes[1].set_xlabel("feature")
ks = sorted(sk.cv_scores_); axes[2].plot(ks, [sk.cv_scores_[k] for k in ks], "o-"); axes[2].axvline(sk.k_, color="r", ls="--")
axes[2].set_title(f"stat_k_selection LOO loss, chosen k={sk.k_}"); axes[2].set_xlabel("k")
plt.tight_layout(); plt.show()
print(f"relevant-feature weight share  classification: {lm.metric_diag_[:2].sum() / lm.metric_diag_.sum():.2f}   "
      f"regression: {lmr.metric_diag_[:2].sum() / lmr.metric_diag_.sum():.2f}   (uniform would be 0.10)")
print(f"prototype_compressed keeps {pc.n_prototypes_} of {len(ytr)} training rows")

## 5. Summary and limitations

In [ ]:
for task, metric, smaller in (("classification", "accuracy", False), ("regression", "mse", True)):
    t = res[res.task == task].groupby(["dataset", "model"])[metric].mean().unstack("model")[ORDER]
    pick = t.idxmin(axis=1) if smaller else t.idxmax(axis=1)
    print(f"best {metric} per dataset:", pick.to_dict())

**What to read from the tables.** In our local 2-seed run the learned diagonal metric gave a large gain for *regression* on the synthetic set (MSE ≈ 0.14 vs 0.56 for the baseline, relevant features up-weighted) but did **not** help *classification* there (log-loss worse than the baseline; the radial class boundary with 10 % label noise is hard for a diagonal NCA objective, and the learned weights even down-weighted the two relevant features). `prototype_compressed` and `stat_k_selection` improved log-loss on the noisy synthetic classification set, while on breast cancer `base`/sklearn were as good or better.

**Honest limitations**
- Two seeds here (3 in the paper) and four datasets: most differences are within one SD, and the real datasets are small. Runtime figures are single un-warmed fits on shared hardware — indicative only. The StatCore variants are pure NumPy with $O(nq)$ distance matrices and are not expected to be faster than sklearn's KD-tree/ball-tree implementation.
- The metric is learned on a random sub-sample of $\le400$ points, is diagonal only (no feature interactions), and the NCA/soft-neighbour objective is non-convex (only a local optimum is reached; no restarts).
- Prototype compression is a heuristic: with a high `compression` ratio it is lossless, with a low ratio it can blur decision boundaries and noisy minority classes.
- `stat_k_selection` selects $k$ on the training data with leave-one-out, so it adds no information on shift; the $10^{-3}$ log-loss tiebreak is a heuristic.
- Probabilities from k-NN vote fractions are coarse (multiples of $1/k$); ECE/log-loss are therefore noisy and not directly comparable to smooth models.